# Module 06 Lab — Cross-System Communication: LangGraph → CrewAI

**Course:** AI Agents Teaching Platform  
**Module:** 06 — Multi-Agent Systems  
**Estimated time:** ~2 hours

---

### What you'll build

A two-system pipeline with a typed Pydantic boundary:

1. **System A (LangGraph)** — research pipeline: planner → researcher → critic
2. **`ResearchHandoff`** — Pydantic model that validates data at the system boundary
3. **System B (CrewAI)** — production pipeline: technical_writer → editor → QA_reviewer

### Why this matters

Real production systems often combine multiple agent frameworks. The type boundary prevents
silent failures where System A produces malformed data that System B silently ignores.

> **Note:** Uses `langchain-anthropic` for both LangGraph and CrewAI.

In [ ]:
%pip install -q langgraph langchain-anthropic crewai pydantic

In [ ]:
import os
from getpass import getpass

os.environ["ANTHROPIC_API_KEY"] = getpass("Paste your Anthropic API key: ")
print("Key set ✓")

## Step 1 — Define the boundary contract (TODO)

The `ResearchHandoff` Pydantic model is the contract between System A and System B.
Both sides must agree on this schema — any validation failure is caught here, not silently propagated.

**TODO:** Add Pydantic validators to enforce:
- `topic` must be non-empty
- `findings` must have at least 2 items
- `confidence` must be between 0.0 and 1.0
- `word_count_target` must be between 100 and 2000

In [ ]:
from pydantic import BaseModel, Field, field_validator
from typing import List

class ResearchHandoff(BaseModel):
    """Typed boundary between the research system and the writing system."""
    topic: str = Field(description="The research topic")
    findings: List[str] = Field(description="Key research findings (at least 2)")
    confidence: float = Field(description="Research confidence score 0.0-1.0")
    word_count_target: int = Field(default=300, description="Target word count for the article")
    requires_expert_review: bool = Field(default=False)

    # TODO: add @field_validator for topic (must not be empty)
    # TODO: add @field_validator for findings (must have >= 2 items)
    # TODO: add @field_validator for confidence (must be 0.0–1.0)
    # TODO: add @field_validator for word_count_target (must be 100–2000)

    model_config = {"str_strip_whitespace": True}


# Quick boundary test — these should pass
handoff = ResearchHandoff(
    topic="Quantum computing fundamentals",
    findings=["Qubits exploit superposition", "Entanglement enables parallelism"],
    confidence=0.85,
    word_count_target=350,
)
print("Valid handoff created ✓")

# These should raise ValidationError once you add validators
import json
invalid_cases = [
    {"topic": "",          "findings": ["f1", "f2"], "confidence": 0.5, "word_count_target": 300},
    {"topic": "topic",     "findings": ["only one"], "confidence": 0.5, "word_count_target": 300},
    {"topic": "topic",     "findings": ["f1", "f2"], "confidence": 1.5, "word_count_target": 300},
    {"topic": "topic",     "findings": ["f1", "f2"], "confidence": 0.5, "word_count_target": 50},
]
from pydantic import ValidationError
passed = 0
for case in invalid_cases:
    try:
        ResearchHandoff(**case)
        print(f"  MISSED invalid case: {case}")
    except ValidationError:
        passed += 1
print(f"Validation caught {passed}/4 invalid cases {'✓' if passed == 4 else '— add validators above'}")

<details>
<summary>💡 Stuck? Reveal the validator solutions</summary>

```python
@field_validator("topic")
@classmethod
def topic_not_empty(cls, v):
    if not v.strip():
        raise ValueError("topic must not be empty")
    return v

@field_validator("findings")
@classmethod
def at_least_two_findings(cls, v):
    if len(v) < 2:
        raise ValueError("at least 2 findings required")
    return v

@field_validator("confidence")
@classmethod
def confidence_range(cls, v):
    if not 0.0 <= v <= 1.0:
        raise ValueError(f"confidence must be 0.0-1.0, got {v}")
    return v

@field_validator("word_count_target")
@classmethod
def word_count_bounds(cls, v):
    if not 100 <= v <= 2000:
        raise ValueError(f"word_count_target must be 100-2000, got {v}")
    return v
```

</details>

## Step 2 — System A: LangGraph research pipeline

Three nodes: planner → researcher → critic.
The final node produces a `ResearchHandoff` object that gets validated at the boundary.

In [ ]:
from typing import TypedDict, Optional
from langgraph.graph import StateGraph, END
from langchain_anthropic import ChatAnthropic
import json

llm = ChatAnthropic(model="claude-haiku-4-5-20251001", max_tokens=512)

class ResearchState(TypedDict):
    question: str
    research_plan: list
    raw_findings: list
    handoff: Optional[ResearchHandoff]


def plan(state: ResearchState) -> dict:
    response = llm.invoke([{"role": "user", "content": (
        f"Create a 3-step research plan for: {state['question']}\n"
        'Return JSON: {"steps": ["step 1", "step 2", "step 3"]}'
    )}])
    try:
        return {"research_plan": json.loads(response.content)["steps"]}
    except (json.JSONDecodeError, KeyError):
        return {"research_plan": [response.content]}


def research(state: ResearchState) -> dict:
    plan_text = "\n".join(f"{i+1}. {s}" for i, s in enumerate(state["research_plan"]))
    response = llm.invoke([{"role": "user", "content": (
        f"Execute this research plan and provide findings:\n{plan_text}\n\n"
        f"Topic: {state['question']}\n"
        'Return JSON: {"findings": ["finding 1", "finding 2", "finding 3"], "confidence": 0.8}'
    )}])
    try:
        data = json.loads(response.content)
        return {"raw_findings": data.get("findings", [response.content])}
    except json.JSONDecodeError:
        return {"raw_findings": [response.content]}


def critique_and_package(state: ResearchState) -> dict:
    response = llm.invoke([{"role": "user", "content": (
        f"Review these findings and provide a confidence score 0.0-1.0:\n"
        f"{json.dumps(state['raw_findings'])}\n"
        'Return JSON: {"improved_findings": [...], "confidence": 0.0-1.0, "requires_expert_review": false}'
    )}])
    try:
        data = json.loads(response.content)
        # Build and validate the handoff — ValidationError propagates if schema is violated
        handoff = ResearchHandoff(
            topic=state["question"],
            findings=data.get("improved_findings", state["raw_findings"]),
            confidence=float(data.get("confidence", 0.7)),
            word_count_target=350,
            requires_expert_review=bool(data.get("requires_expert_review", False)),
        )
    except Exception as e:
        print(f"Boundary validation failed: {e}")
        # Fallback: build minimal valid handoff
        findings = state["raw_findings"][:3] or ["finding 1", "finding 2"]
        if len(findings) < 2:
            findings.append("Additional context needed")
        handoff = ResearchHandoff(
            topic=state["question"], findings=findings,
            confidence=0.5, word_count_target=300,
        )
    return {"handoff": handoff}


graph = StateGraph(ResearchState)
graph.add_node("plan",    plan)
graph.add_node("research", research)
graph.add_node("critique", critique_and_package)
graph.set_entry_point("plan")
graph.add_edge("plan",    "research")
graph.add_edge("research", "critique")
graph.add_edge("critique", END)

system_a = graph.compile()
print("System A (LangGraph) compiled ✓")

## Step 3 — System B: CrewAI writing pipeline (TODO)

Three agents consume the validated `ResearchHandoff` and produce a polished article.

**TODO:** Create the `editor` agent and the `qa_review_task`.

In [ ]:
from crewai import Agent, Task, Crew, Process

technical_writer = Agent(
    role="Technical Writer",
    goal="Transform research findings into a clear, engaging technical article.",
    backstory="Experienced in translating complex research into accessible prose for specialist audiences.",
    llm=llm, verbose=False, allow_delegation=False,
)

editor = Agent(
    role="TODO",
    goal="TODO",
    backstory="TODO",
    llm=llm, verbose=False, allow_delegation=False,
)

qa_reviewer = Agent(
    role="QA Reviewer",
    goal="Verify factual accuracy, completeness, and adherence to the original research findings.",
    backstory="Detail-oriented reviewer who catches inconsistencies between source research and final article.",
    llm=llm, verbose=False, allow_delegation=False,
)

print("System B agents defined (editor TODO) ✓")


def run_system_b(handoff: ResearchHandoff) -> str:
    findings_text = "\n".join(f"- {f}" for f in handoff.findings)

    write_task = Task(
        description=f"""Write a {handoff.word_count_target}-word article on '{handoff.topic}'.
Use these research findings:\n{findings_text}
Research confidence: {handoff.confidence:.0%}{'  [EXPERT REVIEW REQUIRED]' if handoff.requires_expert_review else ''}
Structure: Introduction → Findings → Implications → Conclusion""",
        expected_output=f"A {handoff.word_count_target}-word technical article in markdown.",
        agent=technical_writer,
    )

    edit_task = Task(
        description="Edit the draft for clarity, flow, and precision. Fix any jargon or unclear phrasing.",
        expected_output="The edited article with tracked improvements noted as inline comments.",
        agent=editor,
        context=[write_task],
    )

    # TODO: create qa_review_task
    # description: verify all findings are represented, check confidence caveats, flag any hallucinations
    # expected_output: the final approved article with a QA sign-off note
    # agent: qa_reviewer
    # context: [write_task, edit_task]

    crew = Crew(
        agents=[technical_writer, editor, qa_reviewer],
        tasks=[write_task, edit_task],  # TODO: add qa_review_task
        process=Process.sequential,
        verbose=False,
    )
    return str(crew.kickoff())


print("run_system_b() defined ✓")

<details>
<summary>💡 Stuck? Reveal the editor and qa_task solutions</summary>

```python
editor = Agent(
    role="Senior Editor",
    goal="Improve article clarity, flow, and precision without changing factual content.",
    backstory="Former journal editor with 10 years experience in technical publishing.",
    llm=llm, verbose=False, allow_delegation=False,
)

qa_review_task = Task(
    description="""Verify the edited article against the original research findings:
1. Are all key findings represented?
2. Are confidence caveats included where confidence < 0.7?
3. Flag any claims that weren't in the original findings (potential hallucinations).
Return the final approved article with a QA sign-off section at the end.""",
    expected_output="The final article with a '## QA Sign-off' section listing any issues found.",
    agent=qa_reviewer,
    context=[write_task, edit_task],
)
```

</details>

## Step 4 — Run the full cross-system pipeline

In [ ]:
QUESTION = "What are the key principles of effective prompt engineering for large language models?"

print("=== System A: Research Pipeline (LangGraph) ===")
a_result = system_a.invoke({"question": QUESTION})
handoff = a_result["handoff"]

print(f"Handoff validated ✓")
print(f"  Topic: {handoff.topic}")
print(f"  Findings: {len(handoff.findings)} items")
print(f"  Confidence: {handoff.confidence:.0%}")
print(f"  Expert review needed: {handoff.requires_expert_review}")

print("\n=== System B: Writing Pipeline (CrewAI) ===")
article = run_system_b(handoff)

print("\n--- Final Article ---")
print(article)

## Step 5 — Exercises

### Exercise 1: Inject a validation failure
In `critique_and_package`, force `findings=[]`. Does the boundary catch it?
Does the fallback handler produce a valid handoff?

### Exercise 2: Expert review gate
Add a conditional in the pipeline: if `handoff.requires_expert_review is True`,
print a warning and ask the user to confirm before running System B.

### Exercise 3: Schema evolution
Add a new field `methodology: str` to `ResearchHandoff` with no default.
What breaks? Fix it by adding `methodology: str = "literature_review"` as a default.

### Exercise 4: Replace System B
Instead of CrewAI, write a single LangGraph graph that does the same
write → edit → qa flow. The handoff schema doesn't change — only the consumer changes.

In [ ]:
# Scratch cell — use for exercises


## Step 6 — Reflection

1. What is the blast radius of a validation failure at the boundary? Compare to silent type coercion.
2. Why is `requires_expert_review: bool` a boundary field rather than a System B internal decision?
3. What would break if you passed `state` directly from LangGraph to CrewAI instead of using `ResearchHandoff`?
4. The fallback in `critique_and_package` ensures System B always runs. Is that always the right design?

*(Double-click to edit)*

1. 
2. 
3. 
4. 